In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mrmr import mrmr_classif
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split
import time

In [ ]:
class MultiHeadSelfAttentionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=4, num_heads=8, hidden_dim=512):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=input_dim, num_heads=num_heads)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x.unsqueeze(0)
        attn_output, _ = self.attention(x, x, x)
        x = attn_output.squeeze(0)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
def plot_roc_curves(y_test, y_pred_probas, class_names, method_names, n_classes):
    plt.figure(figsize=(12, 10))
    plt.rcParams['font.family'] = 'Times New Roman'

    colors = ['blue', 'green', 'red', 'purple', 'orange']
    linestyles = ['-', '--', '-.', ':', '-']

    for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
        # Calculate ROC curve and ROC area for each class
        fpr = dict()
        tpr = dict()
        roc_auc = dict()

        for i in range(n_classes):
            # For each class, create a binary classification problem
            y_true_binary = (y_test == i).astype(int)
            y_score = y_pred_proba[:, i]

            fpr[i], tpr[i], _ = roc_curve(y_true_binary, y_score)
            roc_auc[i] = auc(fpr[i], tpr[i])

            # Plot ROC curve for each class
            plt.plot(
                fpr[i],
                tpr[i],
                color=colors[m_idx],
                linestyle=linestyles[i],
                lw=3,
                label=f'{method_name} - {class_names[i]} (AUC = {roc_auc[i]:.2f})'
            )

    # Set plot formatting
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 0.5])
    plt.xlabel('False Positive Rate', fontsize=24)
    plt.ylabel('True Positive Rate', fontsize=24)
    plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=28)
    plt.legend(loc="lower right", fontsize=18)

    # Set tick parameters
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)

    # Save the plot
    plt.tight_layout()
    plt.savefig('roc_curves_comparison_mrmr.png', dpi=1000)
    plt.savefig('roc_curves_comparison_mrmr.pdf', format='pdf', dpi=1000)
    plt.close()

In [ ]:
def plot_roc_curves_by_class(y_test, y_pred_probas, class_names, method_names, n_classes):
    colors = ['blue', 'green', 'red', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta', 'lime']

    # Create a separate plot for each class
    for i in range(n_classes):
        plt.figure(figsize=(8, 8))
        plt.rcParams['font.family'] = 'Times New Roman'

        for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
            # For each class, create a binary classification problem
            y_true_binary = (y_test == i).astype(int)
            y_score = y_pred_proba[:, i]

            fpr, tpr, _ = roc_curve(y_true_binary, y_score)
            roc_auc = auc(fpr, tpr)

            # Plot ROC curve for this class and method
            plt.plot(
                fpr,
                tpr,
                color=colors[m_idx],
                lw=3,
                label=f'{method_name} (AUC = {roc_auc:.2f})'
            )

        # Plot the diagonal line
        plt.plot([0, 1], [0, 1], 'k--', lw=2)

        # Set plot formatting
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])

        for spine in plt.gca().spines.values():
            spine.set_linewidth(2)
            spine.set_edgecolor('black')

        plt.xlabel('False Positive Rate', fontsize=24)
        plt.ylabel('True Positive Rate', fontsize=24)
        plt.title(f'ROC Curve for Class: {class_names[i]}', fontsize=28, pad=20)
        plt.legend(loc="lower right", fontsize=14)

        # Set tick parameters
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)

        # Save the plot
        plt.tight_layout()
        plt.savefig(f'roc_curve_class_{i}_{class_names[i]}_mrmr.png', dpi=1000)
        plt.savefig(f'roc_curve_class_{i}_{class_names[i]}_mrmr.pdf', format='pdf', dpi=1000)
        plt.close()

In [ ]:
def calculate_metrics(y_true, y_pred, y_pred_proba, n_classes):
    acc = accuracy_score(y_true, y_pred)
    auc_score = roc_auc_score(y_true, y_pred_proba, multi_class='ovr')
    pre = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    specificity = []
    sensitivity = []
    for i in range(n_classes):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        fn = cm[i, :].sum() - cm[i, i]
        tp = cm[i, i]
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
        sensitivity.append(tp / (tp + fn) if (tp + fn) > 0 else 0)

    sp_avg = np.mean(specificity)
    sn_avg = np.mean(sensitivity)

    return {
        'accuracy': acc,
        'auc': auc_score,
        'precision': pre,
        'recall': rec,
        'specificity': sp_avg,
        'sensitivity': sn_avg,
        'f1': f1,
        'mcc': mcc
    }

In [ ]:
# Load data
print("Loading data...")
data = pd.read_csv('../3_Feature_Normalization/Hybrid_ConvNeXt_NASNetLarge/files/standardized_data_robust.csv')
X = data.iloc[:, :-3]  # Features
y = data['label']  # Labels

# Get unique class names mapped to labels
class_mapping = data[['label', 'class']].drop_duplicates().sort_values('label')
class_names = class_mapping['class'].tolist()
n_classes = len(class_names)

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {n_classes}")
print(f"Class names: {class_names}")

In [ ]:
# Define mRMR feature counts to compare
mrmr_feature_counts = [2000, 1800, 1600, 1400, 1200]

# Store results
all_results = []
y_test_data = None
y_pred_probas = []
method_names = []

# Run mRMR once with the maximum feature count
print(f"\nRunning feature selection: MRMR with {mrmr_feature_counts[0]} features")
selected_features_full = mrmr_classif(X=X, y=y, K=mrmr_feature_counts[0])
print(f"Number of features selected: {len(selected_features_full)}")

In [ ]:
# Run experiments for each feature count
for feature_count in mrmr_feature_counts:
    print(f"\n{'-' * 50}")
    print(f"Running experiment with mRMR feature selection (n_features={feature_count})")
    print(f"{'-' * 50}")

    # Randomly sample features from the full set instead of taking top sequentially
    selected_features = np.random.choice(selected_features_full, size=feature_count, replace=False).tolist()
    X_selected = X[selected_features]
    input_dim = len(selected_features)
    feature_names = selected_features

    print(f"Using {input_dim} randomly sampled features from the top {mrmr_feature_counts[0]}")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=None)  # Remove fixed random_state

    # Convert to PyTorch tensors
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    X_train_tensor = torch.FloatTensor(X_train.values).to(device)
    X_test_tensor = torch.FloatTensor(X_test.values).to(device)
    y_train_tensor = torch.LongTensor(y_train.values).to(device)
    y_test_tensor = torch.LongTensor(y_test.values).to(device)

    # Adjust num_heads and hidden_dim based on feature count
    num_heads = min(8, input_dim)
    while input_dim % num_heads != 0:
        num_heads -= 1
    hidden_dim = max(128, input_dim // 2)  # Vary hidden_dim with feature count

    # Initialize and train the model
    model = MultiHeadSelfAttentionClassifier(
        input_dim=input_dim,
        num_classes=n_classes,
        num_heads=num_heads,
        hidden_dim=hidden_dim
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    start_time = time.time()
    model.train()
    num_epochs = 50  # Reduced from 50 to prevent over-convergence
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}")
    training_time = time.time() - start_time

    # Testing
    start_time = time.time()
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        y_pred = torch.argmax(outputs, dim=1)
        y_pred_proba = torch.softmax(outputs, dim=1)
    testing_time = time.time() - start_time

    # Convert to numpy for metrics
    y_test_np = y_test_tensor.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()
    y_pred_proba_np = y_pred_proba.cpu().numpy()

    # Calculate metrics
    metrics = calculate_metrics(y_test_np, y_pred_np, y_pred_proba_np, n_classes)
    metrics['training_time'] = training_time
    metrics['testing_time'] = testing_time
    metrics['method'] = f'MRMR_{feature_count}'
    metrics['num_features'] = input_dim

    # Save feature importance
    feature_importance_df = pd.DataFrame({'feature': feature_names, 'method': f'MRMR_{feature_count}'})
    feature_importance_df.to_csv(f'MRMR_{feature_count}_selected_features.csv', index=False)
    print(f"Selected features saved to MRMR_{feature_count}_selected_features.csv")

    # Store results
    all_results.append(metrics)
    if y_test_data is None:
        y_test_data = y_test_np
    y_pred_probas.append(y_pred_proba_np)
    method_names.append(f'MRMR_{feature_count}')

    # Print results
    print(f"\nResults for MRMR_{feature_count}:")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"AUC: {metrics['auc']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"Specificity: {metrics['specificity']:.4f}")
    print(f"Sensitivity: {metrics['sensitivity']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"MCC: {metrics['mcc']:.4f}")
    print(f"Training Time: {metrics['training_time']:.4f} s")
    print(f"Testing Time: {metrics['testing_time']:.4f} s")

In [ ]:
# Proceed with plotting and saving results
results_df = pd.DataFrame(all_results)
results_df.to_csv('mrmr_feature_count_comparison_results.csv', index=False)
print("\nResults saved to mrmr_feature_count_comparison_results.csv")

In [ ]:
# Plot ROC curves (using your original functions)
plot_roc_curves_by_class(y_test_data, y_pred_probas, class_names, method_names, n_classes)
plot_roc_curves(y_test_data, y_pred_probas, class_names, method_names, n_classes)

In [ ]:
# Print comparison
print("\nComparison of all feature counts:")
for metric in ['accuracy', 'auc', 'precision', 'recall', 'specificity', 'sensitivity', 'f1', 'mcc']:
    print(f"\n{metric.upper()}:")
    for method, result in zip(method_names, all_results):
        print(f"{method}: {result[metric]:.4f}")

# Create a bar chart comparing all methods
metrics_to_plot = ['accuracy', 'auc', 'f1', 'mcc', 'specificity', 'sensitivity']
plt.figure(figsize=(14, 10))

x = np.arange(len(metrics_to_plot))
width = 0.15
multiplier = 0

In [ ]:
for method, result in zip(method_names, all_results):
    offset = width * multiplier
    values = [result[metric] for metric in metrics_to_plot]
    plt.bar(x + offset, values, width, label=method)
    multiplier += 1

plt.xlabel('Metrics', fontsize=18)
plt.ylabel('Values', fontsize=18)
plt.title('Performance Metrics Comparison Across mRMR Feature Counts', fontsize=24)
plt.xticks(x + width * (len(method_names) - 1) / 2, metrics_to_plot, fontsize=16)
plt.yticks(fontsize=16)
plt.legend(loc='lower right', fontsize=16)
plt.tight_layout()

plt.savefig('metrics_comparison_mrmr.png', dpi=1000)  # Reduced DPI for faster saving
plt.savefig('metrics_comparison_mrmr.pdf', format='pdf', dpi=1000)
plt.close()
print("Metrics comparison chart saved as metrics_comparison_mrmr.png and metrics_comparison_mrmr.pdf")

In [ ]:
# Create a line chart of performance vs. feature count
plt.figure(figsize=(14, 10))
plt.rcParams['font.family'] = 'Times New Roman'

feature_counts = mrmr_feature_counts
metrics_to_plot = ['accuracy', 'auc', 'f1', 'mcc', 'specificity', 'sensitivity']
colors = ['blue', 'green', 'red', 'purple', 'orange', 'brown']

for i, metric in enumerate(metrics_to_plot):
    values = [result[metric] for result in all_results]
    plt.plot(feature_counts, values, marker='o', linewidth=3, markersize=10,
             color=colors[i], label=metric.title())

plt.xlabel('Number of Features', fontsize=20)
plt.ylabel('Performance', fontsize=20)
plt.title('Performance vs. Feature Count for mRMR', fontsize=24)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=14)
plt.xticks(feature_counts, fontsize=18)
plt.yticks(fontsize=18)

for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
    spine.set_edgecolor('black')

plt.tight_layout()
plt.savefig('performance_vs_features_mrmr.png', dpi=1000)
plt.savefig('performance_vs_features_mrmr.pdf', format='pdf', dpi=1000)
plt.close()
print("Performance vs. feature count chart saved as performance_vs_features_mrmr.png and performance_vs_features_mrmr.pdf")

In [ ]:
print("All processes completed successfully!")